In [2]:
import torch
import torchaudio
import torchaudio.functional as F
import torchaudio.transforms as T

In [3]:
import time
import librosa
import zipfile
import mutagen
import mutagen.wave
import numpy as np
import pandas as pd
import librosa.display
import IPython.display
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from PIL import Image

In [4]:
import pandas as pd
import os

# 1. Caricare il meta file di ESC-50
esc50_path = "ESC-50-master"
esc50_meta = pd.read_csv(os.path.join(esc50_path, "meta", "esc50.csv"))

# Aggiungiamo la colonna dataset
esc50_meta["dataset"] = "ESC-50"

# Il file `filename` è direttamente usato. Possiamo anche costruire il percorso completo:
esc50_meta["filepath"] = esc50_meta["filename"].apply(lambda x: os.path.join(esc50_path, "audio", x))

# Per coerenza, rinominiamo alcune colonne
esc50_meta.rename(columns={
    "category": "class",
}, inplace=True)


# 2. Caricare il meta file di UrbanSound8K
us8k_path = "UrbanSound8K"
us8k_meta = pd.read_csv(os.path.join(us8k_path, "metadata", "UrbanSound8K.csv"))

# Aggiungiamo la colonna dataset
us8k_meta["dataset"] = "UrbanSound8K"

# Costruiamo il percorso completo al file audio
def us8k_filepath(row):
    # fold: row["fold"]
    # filename: row["slice_file_name"]
    return os.path.join(us8k_path, "audio", f"fold{row['fold']}", row["slice_file_name"])

us8k_meta["filepath"] = us8k_meta.apply(us8k_filepath, axis=1)

# Rinominiamo la colonna `class` se si chiama diversamente (qui è proprio `class` già definita)
# Qui la colonna con il nome della classe è "class", mentre quella numerica è "classID"
us8k_meta.rename(columns={
    "class": "class"
}, inplace=True)


# 3. Creare meta file per Emergency Traffic Noise se non esiste
etn_path_traffic = "keggle_with_traffic_noise/traffic/traffic"
etn_files_traffic = [f for f in os.listdir(etn_path_traffic) if f.lower().endswith(".wav")]

etn_meta_traffic = pd.DataFrame(etn_files_traffic, columns=["filename"])
etn_meta_traffic["dataset"] = "emergency_traffic_noise"
etn_meta_traffic["class"] = "traffic"  # se tutti i file appartengono a questa classe
etn_meta_traffic["filepath"] = etn_meta_traffic["filename"].apply(lambda x: os.path.join(etn_path_traffic, x))


etn_path_ambulance = "keggle_with_traffic_noise/Dataset/Dataset/ambulance"
etn_files_ambulance = [f for f in os.listdir(etn_path_ambulance) if f.lower().endswith(".wav")]
etn_meta_ambulance = pd.DataFrame(etn_files_ambulance, columns=["filename"])
etn_meta_ambulance["dataset"] = "emergency_traffic_noise"
etn_meta_ambulance["class"] = "ambulance"  # se tutti i file appartengono a questa classe
etn_meta_ambulance["filepath"] = etn_meta_ambulance["filename"].apply(lambda x: os.path.join(etn_path_ambulance, x))


etn_path_firetruck = "keggle_with_traffic_noise/Dataset/Dataset/firetruck"
etn_files_firetruck = [f for f in os.listdir(etn_path_firetruck) if f.lower().endswith(".wav")]
etn_meta_firetruck = pd.DataFrame(etn_files_firetruck, columns=["filename"])
etn_meta_firetruck["dataset"] = "emergency_traffic_noise"
etn_meta_firetruck["class"] = "firetruck"  # se tutti i file appartengono a questa classe
etn_meta_firetruck["filepath"] = etn_meta_firetruck["filename"].apply(lambda x: os.path.join(etn_path_firetruck, x))

etn_path_police = "keggle_with_traffic_noise/Dataset/Dataset/police"
etn_files_police = [f for f in os.listdir(etn_path_police) if f.lower().endswith(".wav")]
etn_meta_police = pd.DataFrame(etn_files_police, columns=["filename"])
etn_meta_police["dataset"] = "emergency_traffic_noise"
etn_meta_police["class"] = "police"  # se tutti i file appartengono a questa classe
etn_meta_police["filepath"] = etn_meta_police["filename"].apply(lambda x: os.path.join(etn_path_police, x))

# 4. Unione dei DataFrame
combined = pd.concat([esc50_meta[["filepath", "class", "dataset"]],
                      us8k_meta[["filepath", "class", "dataset"]],
                      etn_meta_traffic[["filepath", "class", "dataset"]],
                      etn_meta_ambulance[["filepath", "class", "dataset"]],
                      etn_meta_firetruck[["filepath", "class", "dataset"]],
                      etn_meta_police[["filepath", "class", "dataset"]]],
                     ignore_index=True)

# 5. Salvare in un unico CSV
combined.to_csv("combined_dataset.csv", index=False)


In [5]:
def rms(tensor: torch.Tensor) -> float:
    """Calcola il valore RMS (Root Mean Square) di un segnale audio [channels, time]."""
    return torch.sqrt(torch.mean(tensor**2))

def add_noise_to_audio(waveform: torch.Tensor,
                       desired_snr_db: float,
                       noise_type: str = "gaussian") -> torch.Tensor:
    """
    Aggiunge rumore gaussiano bianco al segnale `waveform` per ottenere il SNR desiderato.
    waveform ha forma [channels, time].
    """
    waveform_noisy = waveform.clone()
    # Calcolo RMS segnale
    sig_rms = rms(waveform_noisy)
    if sig_rms < 1e-8:
        # Se il segnale è quasi silenzioso, meglio non alterare
        return waveform_noisy
    
    if noise_type == "gaussian":
        noise = torch.randn_like(waveform_noisy)
    else:
        raise ValueError(f"Tipo di rumore non supportato: {noise_type}")
    
    noise_rms = rms(noise)
    if noise_rms < 1e-8:
        return waveform_noisy
    
    # desired_snr_db = 20 * log10(sig_rms / noise_rms_scalato)
    # => noise_rms_scalato = sig_rms / (10^(snr_db/20))
    desired_noise_rms = sig_rms / (10 ** (desired_snr_db / 20.0))
    scaling_factor = desired_noise_rms / noise_rms
    
    # Applica lo scaling al rumore e lo somma al segnale
    waveform_noisy += noise * scaling_factor
    return waveform_noisy




In [7]:
CSV_FILE = "combined_dataset.csv"          # CSV con una colonna 'filepath'
OUTPUT_DIR = "dataset_augmented_noise"     # Cartella principale già esistente
SNR_LEVELS = [3, 10, 20]                   # dB
OUTPUT_CSV = "augmented_dataset.csv"        # Nuovo CSV di destinazione

df = pd.read_csv(CSV_FILE)
if 'filepath' not in df.columns:
    raise ValueError("Il CSV deve contenere una colonna 'filepath' con il path del file audio.")

augmented_rows = []  # Per accumulare le info da salvare nel nuovo CSV

# Itera su ogni file audio del CSV
for idx, row in df.iterrows():
    audio_path = row['filepath']
    
    # Recupera eventuali altre colonne: 'class', 'dataset' ecc.
    audio_class = row.get('class', None)
    audio_dataset = row.get('dataset', None)
    
    if not os.path.isfile(audio_path):
        print(f"[AVVISO] File non trovato: {audio_path}")
        continue
    
    # Carica l'audio
    waveform, sample_rate = torchaudio.load(audio_path)
    
    # Crea e salva versioni rumorose per ciascun SNR
    for snr in SNR_LEVELS:
        noisy_waveform = add_noise_to_audio(waveform, snr, noise_type="gaussian")
        
        # Costruiamo il nome del file di output
        basename = os.path.basename(audio_path)               # es: "audio.wav"
        filename_no_ext, ext = os.path.splitext(basename)     # es: ("audio", ".wav")
        out_filename = f"{filename_no_ext}_snr{snr}{ext}"
        
        # Es: "dataset_augmented_noise/3db"
        output_subdir = os.path.join(OUTPUT_DIR, f"{snr}db")
        # Non creiamo la cartella, si assume esista già!
        
        output_path = os.path.join(output_subdir, out_filename)
        
        # Salvataggio file audio
        torchaudio.save(output_path, noisy_waveform, sample_rate)
        print(f"Salvato file con SNR={snr} dB: {output_path}")
        
        # Registra nel nuovo CSV
        augmented_rows.append({
            "filepath": output_path,        # nuovo file rumoroso
            "snr": snr,
            "original_filepath": audio_path,
            "class": audio_class,
            "dataset": audio_dataset
        })

# Crea un DataFrame con tutti i file augmented
augmented_df = pd.DataFrame(augmented_rows)

# Salva il nuovo CSV
augmented_df.to_csv(OUTPUT_CSV, index=False)
print(f"Creato il CSV dei file augmented: {OUTPUT_CSV}")


Salvato file con SNR=3 dB: dataset_augmented_noise/3db/1-100032-A-0_snr3.wav
Salvato file con SNR=10 dB: dataset_augmented_noise/10db/1-100032-A-0_snr10.wav
Salvato file con SNR=20 dB: dataset_augmented_noise/20db/1-100032-A-0_snr20.wav
Salvato file con SNR=3 dB: dataset_augmented_noise/3db/1-100038-A-14_snr3.wav
Salvato file con SNR=10 dB: dataset_augmented_noise/10db/1-100038-A-14_snr10.wav
Salvato file con SNR=20 dB: dataset_augmented_noise/20db/1-100038-A-14_snr20.wav
Salvato file con SNR=3 dB: dataset_augmented_noise/3db/1-100210-A-36_snr3.wav
Salvato file con SNR=10 dB: dataset_augmented_noise/10db/1-100210-A-36_snr10.wav
Salvato file con SNR=20 dB: dataset_augmented_noise/20db/1-100210-A-36_snr20.wav
Salvato file con SNR=3 dB: dataset_augmented_noise/3db/1-100210-B-36_snr3.wav
Salvato file con SNR=10 dB: dataset_augmented_noise/10db/1-100210-B-36_snr10.wav
Salvato file con SNR=20 dB: dataset_augmented_noise/20db/1-100210-B-36_snr20.wav
Salvato file con SNR=3 dB: dataset_augmente